# Edge IIoT - Binary Classification


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
import torch

print(torch.__version__) # X.XX.X+cuXXX / If X.XX.X+cpu it won't work
print(torch.cuda.is_available()) # False
print(torch.version.cuda) # None or mismatched version
print(torch.cuda.device_count()) # 0

2.5.1+cu121
True
12.1
1


In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import DATASETS, SEED
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset_clean"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")


--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/ML-EdgeIIoT-dataset_clean.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (152591, 62)


## 1. Undersampling and Balancing

In [4]:
target     = 'Attack_label'
target_str = 'Attack_type'

In [5]:
y_str = df[target_str]
df_no_label = df.drop(columns=[target_str])

In [6]:
MAX_PRESENCE = 0.02

counts = y_str.value_counts()
total_rows = len(y_str)

print(counts)
print("\nTOTAL: ", total_rows)

Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                13307
Ransomware               10923
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
Backdoor                 10195
Vulnerability_scanner    10075
Port_Scanning            10071
XSS                      10051
Password                  9989
DDoS_TCP                  6011
MITM                      1028
Fingerprinting            1001
Name: count, dtype: int64

TOTAL:  152591


In [7]:
print(f"{'Attack Type':<25} | {'Old Count':<15} | {'New Count':<15}\n" + "-"*55)

sampling_strategy = {}
for attack_type, count in counts.items():
    current_presence = count / total_rows

    if current_presence > MAX_PRESENCE:
        new_count = int(total_rows * MAX_PRESENCE)
    else:
        new_count = count

    sampling_strategy[attack_type] = new_count
    print(f"{attack_type:<25} | {count:<15} | {new_count:<15}")


print("-"*55 + f"\n{'TOTAL':<25} | {counts.sum():<15} | {sum(sampling_strategy.values()):<15}")


Attack Type               | Old Count       | New Count      
-------------------------------------------------------
Normal                    | 24301           | 3051           
DDoS_UDP                  | 14498           | 3051           
DDoS_ICMP                 | 13307           | 3051           
Ransomware                | 10923           | 3051           
DDoS_HTTP                 | 10561           | 3051           
SQL_injection             | 10311           | 3051           
Uploading                 | 10269           | 3051           
Backdoor                  | 10195           | 3051           
Vulnerability_scanner     | 10075           | 3051           
Port_Scanning             | 10071           | 3051           
XSS                       | 10051           | 3051           
Password                  | 9989            | 3051           
DDoS_TCP                  | 6011            | 3051           
MITM                      | 1028            | 1028           
Fingerprinting

In [8]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=SEED)
df_no_label, y_str = rus.fit_resample(df_no_label, y_str)

print(df_no_label.shape)

(41692, 61)


In [9]:
# X/y split
X     = df_no_label.drop(columns=[target])
y     = df_no_label[target]

## 1. Pre-processing

In [10]:
# Select only numeric

print("--- Non-numeric cols to drop ---\n\n", X.select_dtypes(include=['str', 'object', 'category']).columns)

X = X.select_dtypes(include=['number'])

print("\n\nRemaining categorical cols:", len(X.select_dtypes(include=['str', 'object', 'category']).columns))

--- Non-numeric cols to drop ---

 Index(['http.file_data', 'http.referer', 'http.request.path',
       'http.request.uri.query', 'http.request.version', 'mqtt.msg', 'proto'],
      dtype='str')


Remaining categorical cols: 0


In [11]:
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")

NaN values in target variable: 0
NaN values in features: 0


In [12]:
X_train, X_test, y_train, y_test, y_str_train, y_str_test = train_test_split(
    X, y, y_str, test_size=0.2, random_state=SEED, stratify=y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (33353, 53)
X_test shape: (8339, 53)


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

## 2. LazyPredict
[Docs](https://pypi.org/project/lazypredict/)

In [14]:
from lazypredict.Supervised import LazyClassifier

# With categorical encoding, timeout, cross-validation, and GPU
clf = LazyClassifier(
    verbose=1,                          # Show progress
    ignore_warnings=True,               # Suppress warnings
    custom_metric=None,                 # Use default metrics
    predictions=False,                  # Don't Return predictions
    classifiers='all',                  # Use all available classifiers
    timeout=60,                         # Max time per model in seconds
    cv=5,                               # Cross-validation folds (optional)
)

models, _ = clf.fit(X_train, X_test, y_train, y_test)
print("\n--- Models Evaluated ---")

  0%|          | 0/32 [00:00<?, ?it/s]

/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1


--- Models Evaluated ---


In [15]:
display(models)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.997362,0.988762,0.999839,0.997358,0.997355,0.997362,0.998141,0.000451,0.991828,0.001795,0.999884,0.000085,0.998139,0.000450,0.998139,0.000452,0.998141,0.000451,352.905489
XGBClassifier,0.997002,0.987813,0.999766,0.996999,0.996996,0.997002,0.997751,0.000502,0.990109,0.002052,0.999823,0.000099,0.997748,0.000502,0.997747,0.000502,0.997751,0.000502,2.276736
ExtraTreesClassifier,0.997242,0.987187,0.999852,0.997235,0.997232,0.997242,0.997631,0.000439,0.990421,0.001571,0.999258,0.000862,0.997630,0.000438,0.997629,0.000437,0.997631,0.000439,1.246643
RandomForestClassifier,0.996882,0.984728,0.999251,0.996870,0.996869,0.996882,0.997481,0.000564,0.986945,0.003058,0.999334,0.000813,0.997471,0.000568,0.997473,0.000567,0.997481,0.000564,2.914623
ExtraTreeClassifier,0.995923,0.982701,0.982701,0.995913,0.995908,0.995923,0.995563,0.001156,0.982891,0.003369,0.982891,0.003369,0.995561,0.001150,0.995562,0.001145,0.995563,0.001156,1.431946
BaggingClassifier,0.996283,0.982140,0.993105,0.996267,0.996264,0.996283,0.996732,0.000320,0.987106,0.002133,0.995822,0.000972,0.996729,0.000322,0.996728,0.000321,0.996732,0.000320,2.187868
DecisionTreeClassifier,0.995203,0.980048,0.980048,0.995192,0.995185,0.995203,0.995802,0.000776,0.983772,0.005363,0.983772,0.005363,0.995798,0.000781,0.995806,0.000782,0.995802,0.000776,1.592900
KNeighborsClassifier,0.985370,0.925669,0.978577,0.985057,0.985047,0.985370,0.986088,0.002180,0.927794,0.015000,0.978315,0.006866,0.985764,0.002321,0.985809,0.002257,0.986088,0.002180,1.452249
AdaBoostClassifier,0.983691,0.912684,0.995759,0.983217,0.983293,0.983691,0.986538,0.002278,0.925201,0.007958,0.996240,0.000233,0.986183,0.002292,0.986339,0.002404,0.986538,0.002278,2.757400


In [16]:
display(models.sort_values(by='F1 Score CV Mean', ascending=False).head(3))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.997362,0.988762,0.999839,0.997358,0.997355,0.997362,0.998141,0.000451,0.991828,0.001795,0.999884,0.000085,0.998139,0.000450,0.998139,0.000452,0.998141,0.000451,352.905489
XGBClassifier,0.997002,0.987813,0.999766,0.996999,0.996996,0.997002,0.997751,0.000502,0.990109,0.002052,0.999823,0.000099,0.997748,0.000502,0.997747,0.000502,0.997751,0.000502,2.276736
ExtraTreesClassifier,0.997242,0.987187,0.999852,0.997235,0.997232,0.997242,0.997631,0.000439,0.990421,0.001571,0.999258,0.000862,0.997630,0.000438,0.997629,0.000437,0.997631,0.000439,1.246643
